# Finding Redundant 1D Subspaces in LLM Activations

This notebook discovers **redundant 1-dimensional subspaces** in language model activations: directions where:
1. Activations have **large projections** (model actively uses this direction)
2. **Removing** this projection has **minimal impact** on output token distributions

In [ ]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [ ]:
import torch
from src.utils.env import set_seed

set_seed(42)

# torch.set_float32_matmul_precision("high")

In [ ]:
import torch
from scripts.utils.load_model import load_model
from src.model import TargetedModel
from src.aliases import Conv

model_name = "meta-llama/Llama-2-7b-chat-hf"

print(f"====== Testing model {model_name} ======")
model, tokenizer = load_model(model_name, dtype="bfloat16")
targeted_model = TargetedModel(model, tokenizer, is_chat=True)
print(f"Model {model_name}")
print(model)

In [ ]:
from scripts.utils.load_dataset import load_dataset
from src.data import TableLoader

ds_train, ds_val, ds_test = load_dataset("hh-rlhf")

dl_train = TableLoader(ds_train, batch_size=16, shuffle=True)
dl_val = TableLoader(ds_val, batch_size=8, shuffle=True)
dl_test = TableLoader(ds_test, batch_size=8, shuffle=True)

In [ ]:
import torch
from tqdm.auto import tqdm

from src.model import TargetedModel
from src.data import TableLoader, TableIterator
from src.activation_extractor import ActivationExtractor, ActivationManipulator
from src.metrics import Metrics
from src.losses import Loss
from src.functional import project, compute_targets_mask
from src.norm.utils import probe_layer_dim, redundancy_score_norm
from src.config import StopCriteria


def train_norm(
    targeted_model: TargetedModel,
    layer: str,
    dl_train: TableLoader,
    stop_criteria: StopCriteria,
    learning_rate: float = 0.01,
    proj_weight: float = 0.1,
) -> tuple[torch.Tensor, list[dict[str, float]]]:

    stop_criteria.reset()
    layer_dim = probe_layer_dim(targeted_model, layer)
    w = torch.randn(layer_dim, device=targeted_model.device, dtype=targeted_model.dtype, requires_grad=True)
    optim = torch.optim.Adam([w], lr=learning_rate)

    best_score = -float("inf")
    best_direction = w.clone().detach()

    def subtract_projection(activations: torch.Tensor) -> torch.Tensor:
        projection = project(activations, direction=w, normalize=True)
        return activations - projection

    extractor = ActivationExtractor(targeted_model.model, layer)
    manipulator = ActivationManipulator(targeted_model.model, layer, manipulation_fn=subtract_projection)

    history = []
    num_steps = min(stop_criteria.max_steps, len(dl_train))
    pbar = tqdm(TableIterator(dl_train, num_batches=num_steps), desc="Training", leave=False)

    for batch in pbar:
        if stop_criteria.should_stop():
            break

        conversations = batch["prompt"]
        encodings = targeted_model.tokenize(conversations)
        targets_mask = compute_targets_mask(encodings)

        optim.zero_grad()
        v = torch.nn.functional.normalize(w, dim=0)

        with torch.no_grad(), extractor.capture():
            baseline_logits = targeted_model.forward(encodings).logits.detach()
            activations = extractor.get_activations()[layer].detach()
            activations_normalized = torch.nn.functional.normalize(activations, dim=-1)

        with manipulator.capture():
            modified_logits = targeted_model.forward(encodings).logits

        kl_div = Loss.kl_divergence(
            baseline_logits=baseline_logits,
            modified_logits=modified_logits,
            targets_mask=targets_mask,
            top_k=None,
        )

        proj_l2_rel = Loss.projection_l2_norm(
            activations=activations_normalized,
            direction=v,
            targets_mask=targets_mask,
            reduction="mean",
        )

        top10_agr = Metrics.topk_agreement(
            baseline_logits=baseline_logits.detach(),
            modified_logits=modified_logits.detach(),
            targets_mask=targets_mask,
            top_k=10,
        )

        top1_acc = Metrics.top1_accuracy(
            baseline_logits=baseline_logits.detach(),
            modified_logits=modified_logits.detach(),
            targets_mask=targets_mask,
        )

        score_val = redundancy_score_norm(
            proj_norm=proj_l2_rel.item(),
            top1_acc=top1_acc,
            top10_agr=top10_agr,
        )

        if score_val > best_score:
            best_score = score_val
            best_direction = v.detach().clone()

        # compute loss and update
        loss = kl_div - proj_weight * proj_l2_rel
        loss.backward()
        optim.step()

        METRICS = {
            "loss": loss.item(),
            "kl_div": kl_div.item(),
            "proj_l2_rel": proj_l2_rel.item(),
            "top1_acc": top1_acc,
            "top10_agr": top10_agr,
            "score": score_val,
            "best_score": best_score,
        }

        # update progress
        history.append(METRICS)
        pbar.set_postfix({k: f"{v:.4f}" for k, v in METRICS.items()})
        stop_criteria.update(value=score_val)

    pbar.close()

    return best_direction, history


In [ ]:
from src.config import StopCriteria

train_norm(
    targeted_model=targeted_model,
    layer="model.layers.0",
    dl_train=dl_train,
    stop_criteria=StopCriteria(max_steps=1000),
)